# Spot detection — N3

Basic results only: per-image spot counts (saved as CSV) and the comparison boxplot (saved as PNG). No inline image display.

Data: 3-channel acquisition (AF647 tubulin, AF488, DAPI). Channel 0 (AF647, tubulin) is used, as in N1/N2.

Pipeline: max-Z projection -> Spotiflow (`general` pretrained model) -> spot count. See `results/README.md`.

In [1]:
from pylibCZIrw import czi as pyczi
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
from spotiflow.model import Spotiflow

model = Spotiflow.from_pretrained("general")


INFO:spotiflow.model.spotiflow:Loading pretrained model: general


In [2]:
# ── Per-image spot counting, CSV table, comparison plot (no image display) ───

def process_directory(directory):
    """Run spot detection on all .czi files; returns list of (filename, n_spots)."""
    results = []
    for filename in sorted(os.listdir(directory)):
        if not filename.endswith(".czi"):
            continue
        file_path = os.path.join(directory, filename)
        with pyczi.open_czi(file_path) as czidoc:
            bbox = czidoc.total_bounding_box
            z_start, z_size = bbox.get('Z', (0, 1))
            t_start = bbox.get('T', (0, 1))[0]
            slices = []
            for z in range(z_start, z_start + z_size):
                plane = czidoc.read(plane={'C': 0, 'Z': z, 'T': t_start})
                slices.append(plane[..., 0].astype(np.float32))
        img_2d = np.max(slices, axis=0) if len(slices) > 1 else slices[0]
        points, details = model.predict(img_2d, verbose=False)
        results.append((filename, len(points)))
    return results


def analyze_all(directories):
    rows = []
    for cond, directory in directories.items():
        for fname, n_spots in process_directory(directory):
            rows.append({'condition': cond, 'filename': fname, 'n_spots': n_spots})
    return pd.DataFrame(rows)


def plot_comparison(df, save_path):
    labels = list(dict.fromkeys(df['condition']))
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    data   = [df.loc[df['condition'] == l, 'n_spots'].to_numpy() for l in labels]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.boxplot(data, labels=labels, widths=0.4)
    rng = np.random.default_rng(0)
    for i, l in enumerate(labels, start=1):
        sub   = df.loc[df['condition'] == l]
        xs    = rng.normal(i, 0.05, size=len(sub))
        color = colors[(i - 1) % len(colors)]
        ax.scatter(xs, sub['n_spots'], s=50, color=color, zorder=3, alpha=0.9)
        for x, y, fname in zip(xs, sub['n_spots'], sub['filename']):
            ax.annotate(os.path.splitext(fname)[0], xy=(x, y), xytext=(6, 0),
                        textcoords='offset points', fontsize=8, va='center', color=color)
    ax.set_ylabel('Number of Spots')
    ax.set_title('Comparison of Spot Counts Across Conditions')
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved comparison plot to {save_path}")


## Directories and run

In [3]:
directories = {
    "bmi": "/Users/ctuna/Desktop/axon_codes/N3/BMI!/id100426 bmi1 j14_data/",
    "Ni": "/Users/ctuna/Desktop/axon_codes/N3/NI/ni n=3 _data/",
    "scr": "/Users/ctuna/Desktop/axon_codes/N3/SCR/scr n=3_data/",
}

RESULTS_DIR = '/Users/ctuna/Desktop/axon_codes/results/N3'
CSV_PATH  = os.path.join(RESULTS_DIR, "csv", "spot_counts_N3.csv")
PLOT_PATH = os.path.join(RESULTS_DIR, "figures", "spot_count_comparison_N3.png")

df = analyze_all(directories)
df.to_csv(CSV_PATH, index=False)
print(f"Saved counts for {len(df)} images to {CSV_PATH}")

plot_comparison(df, PLOT_PATH)
df


Saved counts for 54 images to /Users/ctuna/Desktop/axon_codes/results/N3/csv/spot_counts_N3.csv
Saved comparison plot to /Users/ctuna/Desktop/axon_codes/results/N3/figures/spot_count_comparison_N3.png


/var/folders/wn/2ktvmvf11gg5_vwy7_v2ys6m0000gp/T/ipykernel_86935/448926825.py:37: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=labels, widths=0.4)


,condition,filename,n_spots
0,bmi,New-01.czi,377
1,bmi,New-02.czi,385
2,bmi,New-03.czi,460
3,bmi,New-04.czi,421
4,bmi,New-05.czi,565
5,bmi,New-06.czi,780
6,bmi,New-07.czi,1047
7,bmi,New-08.czi,1154
8,bmi,New-09.czi,1199
9,bmi,New-10.czi,1051
